In [30]:
# # ROLLING /WINDOW FUNCTIONS
# Rolling functions let you compute statistics over a sliding time window — the foundation of anomaly detection, trend analysis, and dynamic alerting in AIOps.

In [31]:
# rolling() 
import pandas as pd
df_metrics = pd.read_csv("server_metrics.csv")
df_tickets = pd.read_csv("incidents.csv")
df_logs = pd.read_csv("app_logs.csv")

In [32]:
df_metrics_sorted = df_metrics.sort_values(["server_id", "timestamp"])
# 3- period rolling mean per server
df_metrics_sorted["cpu_rolling_mean"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x: x.rolling(window=3).mean())
print(df_metrics_sorted["cpu_rolling_mean"])
print(df_metrics_sorted)

0            NaN
500          NaN
5      60.400000
505    71.200000
10     86.866667
         ...    
479    63.866667
484    50.466667
489    41.266667
494    38.166667
499    46.933333
Name: cpu_rolling_mean, Length: 525, dtype: float64
               timestamp server_id  cpu_pct  memory_pct  response_ms  \
0    2026-01-01 00:00:00    srv-01     49.6        94.6        891.8   
500  2026-01-01 00:00:00    srv-01     49.6        94.6        891.8   
5    2026-01-01 01:00:00    srv-01     82.0        43.6        641.4   
505  2026-01-01 01:00:00    srv-01     82.0        43.6        641.4   
10   2026-01-01 02:00:00    srv-01     96.6        82.7       1130.4   
..                   ...       ...      ...         ...          ...   
479  2026-01-04 23:00:00    srv-05     47.3        40.5         79.3   
484  2026-01-05 00:00:00    srv-05     32.6        31.6        919.6   
489  2026-01-05 01:00:00    srv-05     43.9        47.6        273.8   
494  2026-01-05 02:00:00    srv-05     38

In [33]:
# 3-period rolling std - measures volatility
df_metrics_sorted["cpu_rolling_std"] = df_metrics_sorted.groupby("server_id",observed=True)["cpu_pct"].transform(lambda x: x.rolling(window=3).std())
print(df_metrics_sorted[["server_id", "timestamp", "cpu_pct", "cpu_rolling_mean", "cpu_rolling_std"]].head(5))
# AIOps use — rolling mean smooths noise. Rolling std measures how unstable a server is over time.

    server_id            timestamp  cpu_pct  cpu_rolling_mean  cpu_rolling_std
0      srv-01  2026-01-01 00:00:00     49.6               NaN              NaN
500    srv-01  2026-01-01 00:00:00     49.6               NaN              NaN
5      srv-01  2026-01-01 01:00:00     82.0         60.400000        18.706149
505    srv-01  2026-01-01 01:00:00     82.0         71.200000        18.706149
10     srv-01  2026-01-01 02:00:00     96.6         86.866667         8.429314


In [34]:
# min_periods - handling NaN at start of window
# default rolling -first (window-1) rows are Nan
# min_periods=1 fills from first available row
df_metrics_sorted["cpu_rolling_mean"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x:x.rolling(window=3, min_periods=1).mean())
print(df_metrics_sorted[["server_id", "cpu_pct", "cpu_rolling_mean"]].head(5))
# AIOps use — in production you never want NaN in your alert pipeline. min_periods=1 ensures every row has a value.

    server_id  cpu_pct  cpu_rolling_mean
0      srv-01     49.6         49.600000
500    srv-01     49.6         49.600000
5      srv-01     82.0         60.400000
505    srv-01     82.0         71.200000
10     srv-01     96.6         86.866667


In [35]:
# Rolling window for anomaly detection - Z-score method

# Dynamic Z-score using rolling mean and std
df_metrics_sorted = df_metrics.drop_duplicates(subset = ["server_id", "timestamp"]).sort_values(["server_id", "timestamp"])
df_metrics_sorted["cpu_rolling_mean"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x: x.rolling(window=6, min_periods=1).mean())

df_metrics_sorted["cpu_rolling_std"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x: x.rolling(window=6, min_periods=1).std())

df_metrics_sorted["cpu_zscore"] = (
    (df_metrics_sorted["cpu_pct"] - df_metrics_sorted["cpu_rolling_mean"]) /
df_metrics_sorted["cpu_rolling_std"]
)
print(df_metrics_sorted["cpu_zscore"].dtype)
print(df_metrics_sorted["cpu_zscore"].abs().head())

# Flag anomalies - z-score > 2 is a standard threshold

df_metrics_sorted["is_anomaly"] = df_metrics_sorted["cpu_zscore"].abs().fillna(0) > 1
# df_metrics_sorted["is_anomaly"] = df_metrics_sorted["cpu_zscore"].abs() > 2 (GOT NO ANOMALY )
print(f"Anomalies detected: {df_metrics_sorted["is_anomaly"].sum()}")
print(df_metrics_sorted[df_metrics_sorted["is_anomaly"]][["server_id", "timestamp", "cpu_pct", "cpu_rolling_mean", "cpu_zscore"]].head(5))
#print(df_metrics_sorted.head(5))
# AIOps use — static thresholds (cpu > 80) miss context. Dynamic Z-score catches spikes relative to each server's recent baseline — far more accurate.

float64
0          NaN
5     0.707107
10    0.853592
15    0.058506
20    1.461676
Name: cpu_zscore, dtype: float64
Anomalies detected: 160
   server_id            timestamp  cpu_pct  cpu_rolling_mean  cpu_zscore
20    srv-01  2026-01-01 04:00:00     22.5         65.660000   -1.461676
60    srv-01  2026-01-01 12:00:00     24.1         45.133333   -1.226200
65    srv-01  2026-01-01 13:00:00     78.3         52.550000    1.252685
80    srv-01  2026-01-01 16:00:00     28.1         49.166667   -1.014408
90    srv-01  2026-01-01 18:00:00     81.4         59.350000    1.092008


In [36]:
# Rule to remember
# Window                     Threshold                  Behaviour
# ------                    ------------              --------------
# Small (3-6)               Use > 1                   More sensitive
# Large (12-24)             Use > 2                  Industry standard
# Production AIOps          window=24, threshold=2   One full day baseline

# For your dataset size (100 rows per server) — window=12, threshold=2 is the sweet spot.
# window=6 produces small z-scores — threshold > 1 works for this dataset
# Production rule: window=24 (full day baseline), threshold=2
# Smaller dataset → lower threshold; larger dataset → threshold=2 safe

In [37]:
# Rolling window on response_ms - SLA breach detection

In [38]:
# Rolling p95 response time - are we trending toward SLA breach ?
df_metrics_sorted["response_rolling_p95"] = df_metrics_sorted.groupby("server_id", observed=True)["response_ms"].\
transform(lambda x: x.rolling(window=6, min_periods=1).quantile(0.95))
print(df_metrics_sorted)

               timestamp server_id  cpu_pct  memory_pct  response_ms  \
0    2026-01-01 00:00:00    srv-01     49.6        94.6        891.8   
5    2026-01-01 01:00:00    srv-01     82.0        43.6        641.4   
10   2026-01-01 02:00:00    srv-01     96.6        82.7       1130.4   
15   2026-01-01 03:00:00    srv-01     77.6        82.4        135.2   
20   2026-01-01 04:00:00    srv-01     22.5        73.3        411.5   
..                   ...       ...      ...         ...          ...   
479  2026-01-04 23:00:00    srv-05     47.3        40.5         79.3   
484  2026-01-05 00:00:00    srv-05     32.6        31.6        919.6   
489  2026-01-05 01:00:00    srv-05     43.9        47.6        273.8   
494  2026-01-05 02:00:00    srv-05     38.0        95.6       1095.9   
499  2026-01-05 03:00:00    srv-05     58.9        69.3       1045.4   

     disk_pct   status  cpu_rolling_mean  cpu_rolling_std  cpu_zscore  \
0        68.9       ok         49.600000              NaN     

In [39]:
# Flag where rolling p95 exceeds SLA threshold (eg 900 ms)
SLA_THRESHOLD = 900
df_metrics_sorted["sla_risk"] = df_metrics_sorted["response_rolling_p95"] > SLA_THRESHOLD
print(df_metrics_sorted["sla_risk"].count()) # Total nos analysed
print(df_metrics_sorted["sla_risk"].sum()) # only true 
print(df_metrics_sorted[df_metrics_sorted["sla_risk"]][["server_id", "timestamp", "response_ms", 
                                                        "response_rolling_p95"]].head(5))

500
380
   server_id            timestamp  response_ms  response_rolling_p95
10    srv-01  2026-01-01 02:00:00       1130.4               1106.54
15    srv-01  2026-01-01 03:00:00        135.2               1094.61
20    srv-01  2026-01-01 04:00:00        411.5               1082.68
25    srv-01  2026-01-01 05:00:00       1039.8               1107.75
30    srv-01  2026-01-01 06:00:00        216.6               1107.75


In [40]:
# AIOps use — rolling p95 is the industry standard for SLA monitoring. A single slow response is noise — sustained p95 above threshold is a real problem.

In [41]:
# expanding() - cumulative window

In [42]:
# cumulative mean from start - all-time baseline per server
print(df_metrics_sorted)
df_metrics_sorted["cpu_expanding_mean"] = df_metrics_sorted.groupby("server_id", observed=True)\
["cpu_pct"].transform(lambda x:x.expanding().mean())
print(df_metrics_sorted[["server_id", "cpu_pct", "cpu_expanding_mean"]].head(5))

               timestamp server_id  cpu_pct  memory_pct  response_ms  \
0    2026-01-01 00:00:00    srv-01     49.6        94.6        891.8   
5    2026-01-01 01:00:00    srv-01     82.0        43.6        641.4   
10   2026-01-01 02:00:00    srv-01     96.6        82.7       1130.4   
15   2026-01-01 03:00:00    srv-01     77.6        82.4        135.2   
20   2026-01-01 04:00:00    srv-01     22.5        73.3        411.5   
..                   ...       ...      ...         ...          ...   
479  2026-01-04 23:00:00    srv-05     47.3        40.5         79.3   
484  2026-01-05 00:00:00    srv-05     32.6        31.6        919.6   
489  2026-01-05 01:00:00    srv-05     43.9        47.6        273.8   
494  2026-01-05 02:00:00    srv-05     38.0        95.6       1095.9   
499  2026-01-05 03:00:00    srv-05     58.9        69.3       1045.4   

     disk_pct   status  cpu_rolling_mean  cpu_rolling_std  cpu_zscore  \
0        68.9       ok         49.600000              NaN     

In [43]:
# # AIOps use — expanding mean gives you the all-time baseline for a server. Compare rolling mean vs expanding mean to detect long-term drift.
# Key rules
# ---------
# Rule                                   Why
# -----                                  ----
# Always sort_values before rolling      Window must be in time order
# Always groupby before rolling          Per-server window, not fleet-wide
# Use min_periods=1 in production        No NaN in alert pipelines
# Rolling Z-score over static threshold  Context-aware anomaly detection
# Rolling p95 for SLA monitoring         Industry standard metric

In [45]:
# LAB TASKS

In [ ]:
# Task 1 — server_metrics
# Compute 6-period rolling mean and std for cpu_pct per server. 
# Which server has the highest average rolling std — meaning most unstable CPU over time?

In [46]:
df_metrics_sorted1 = df_metrics.sort_values(["server_id", "timestamp"])
df_metrics_sorted1["cpu_pct_roll_mean"] = df_metrics_sorted.groupby("server_id", observed=True)["cpu_pct"].\
    transform(lambda x:x.rolling(window=6).mean())
df_metrics_sorted1["cpu_pct_roll_std"] = df_metrics_sorted.groupby("server_id", observed=True)["cpu_pct"].\
    transform(lambda x:x.rolling(window=6).std())
# print(df_metrics_sorted1)
print(df_metrics_sorted1.idxmax())

timestamp            495
server_id              4
cpu_pct              343
memory_pct           106
response_ms          169
disk_pct             266
status                20
cpu_pct_roll_mean    429
cpu_pct_roll_std     308
dtype: int64


In [ ]:
# Task 2 — server_metrics
# Compute rolling p95 for response_ms per server using window=6. Flag rows where rolling p95 exceeds 900ms as sla_risk. Which server has the most SLA risk readings?

In [49]:
# Sort data first
df_metrics_sorted = df_metrics.sort_values(
        ["server_id", "timestamp"]
)

# Compute rolling p95 response time per server
df_metrics_sorted["response_rolling_p95"] = (
        df_metrics_sorted.groupby(
                    "server_id",
                            observed=True
        )["response_ms"]
            .transform(
                        lambda x: x.rolling(window=6).quantile(0.95)
            )
)

# SLA threshold
SLA_THRESHOLD = 900

# Flag SLA risk rows
df_metrics_sorted["sla_risk"] = (
        df_metrics_sorted["response_rolling_p95"] > SLA_THRESHOLD
)

# Count SLA risk readings per server
sla_counts = df_metrics_sorted.groupby(
        "server_id",
            observed=True
)["sla_risk"].sum()

print("SLA Risk Counts:")
print(sla_counts)

# Server with most SLA risks
worst_server = sla_counts.idxmax()

print("\nServer withmost SLA risk readings:")
print(worst_server)

SLA Risk Counts:
server_id
srv-01    83
srv-02    73
srv-03    67
srv-04    81
srv-05    76
Name: sla_risk, dtype: int64

Server withmost SLA risk readings:
srv-01


In [ ]:
# Task 3 — server_metrics
# Use dynamic Z-score (window=12) on memory_pct per server. Flag anomalies at threshold > 1.5. How many memory anomalies per server?

In [50]:
# Sort data first
df_metrics_sorted = df_metrics.sort_values(
    ["server_id", "timestamp"]
)

# Rolling mean (window=12)
df_metrics_sorted["memory_roll_mean"] = (
    df_metrics_sorted.groupby(
        "server_id",
        observed=True
    )["memory_pct"]
    .transform(
        lambda x: x.rolling(window=12).mean()
    )
)

# Rolling std (window=12)
df_metrics_sorted["memory_roll_std"] = (
    df_metrics_sorted.groupby(
        "server_id",
        observed=True
    )["memory_pct"]
    .transform(
        lambda x: x.rolling(window=12).std()
    )
)

# Dynamic Z-score
df_metrics_sorted["memory_zscore"] = (
    (
        df_metrics_sorted["memory_pct"]
        - df_metrics_sorted["memory_roll_mean"]
    )
    /
    df_metrics_sorted["memory_roll_std"]
)

# Flag anomalies
df_metrics_sorted["memory_anomaly"] = (
    df_metrics_sorted["memory_zscore"].abs() > 1.5
)

# Count anomalies per server
anomaly_counts = df_metrics_sorted.groupby(
    "server_id",
    observed=True
)["memory_anomaly"].sum()

print("Memory anomalies per server:")
print(anomaly_counts)

Memory anomalies per server:
server_id
srv-01    11
srv-02    14
srv-03    10
srv-04     5
srv-05     9
Name: memory_anomaly, dtype: int64


In [ ]:
# Task 4 — server_metrics
# Compute expanding mean for cpu_pct per server. Add a column above_lifetime_avg — flag rows where current cpu_pct exceeds expanding mean. What % of readings are above lifetime average per server?

In [51]:
# Sort data by server and timestamp
df_metrics_sorted = df_metrics.sort_values(
    ["server_id", "timestamp"]
)

# Compute expanding (lifetime) mean CPU per server
df_metrics_sorted["cpu_expanding_mean"] = (
    df_metrics_sorted.groupby(
        "server_id",
        observed=True
    )["cpu_pct"]
    .transform(
        lambda x: x.expanding().mean()
    )
)

# Flag rows where current CPU exceeds lifetime average
df_metrics_sorted["above_lifetime_avg"] = (
    df_metrics_sorted["cpu_pct"]
    >
    df_metrics_sorted["cpu_expanding_mean"]
)

# Compute percentage of readings above lifetime average
above_pct = (
    df_metrics_sorted.groupby(
        "server_id",
        observed=True
    )["above_lifetime_avg"]
    .mean()
    * 100
)

print("Percentage of readings above lifetime average:")
print(above_pct)

Percentage of readings above lifetime average:
server_id
srv-01    50.476190
srv-02    47.619048
srv-03    50.476190
srv-04    52.380952
srv-05    49.523810
Name: above_lifetime_avg, dtype: float64


In [52]:
# Task 5 — app_logs
# Resample log counts by hour. Compute 3-period rolling mean of log count. Flag hours where actual count exceeds rolling mean by more than 2x — these are log storm windows.

In [53]:
# Convert timestamp to datetime
df_logs["timestamp"] = pd.to_datetime(df_logs["timestamp"])

# Set timestamp as index
df_logs_ts = df_logs.set_index("timestamp")

# Resample hourly log counts
hourly_logs = df_logs_ts.resample("h").size()

# Convert to DataFrame
hourly_logs = hourly_logs.to_frame(name="log_count")

# Compute 3-period rolling mean
hourly_logs["rolling_mean_3"] = (
    hourly_logs["log_count"]
    .rolling(window=3)
    .mean()
)

# Flag log storm windows
hourly_logs["log_storm"] = (
    hourly_logs["log_count"]
    >
    hourly_logs["rolling_mean_3"] * 2
)

print(hourly_logs)

# Optional: show only storm windows
storm_windows = hourly_logs[hourly_logs["log_storm"]]

print("\nLog Storm Windows:")
print(storm_windows)

                     log_count  rolling_mean_3  log_storm
timestamp                                                
2026-01-01 00:00:00          2             NaN      False
2026-01-01 01:00:00          2             NaN      False
2026-01-01 02:00:00          1        1.666667      False
2026-01-01 03:00:00          4        2.333333      False
2026-01-01 04:00:00          5        3.333333      False
...                        ...             ...        ...
2026-01-10 19:00:00          0        1.000000      False
2026-01-10 20:00:00          3        1.000000       True
2026-01-10 21:00:00          3        2.000000      False
2026-01-10 22:00:00          0        2.000000      False
2026-01-10 23:00:00          1        1.333333      False

[240 rows x 3 columns]

Log Storm Windows:
                     log_count  rolling_mean_3  log_storm
timestamp                                                
2026-01-02 04:00:00          5        2.333333       True
2026-01-02 10:00:00         

In [ ]:
# Task 6 — cross-dataset
# Find all is_anomaly CPU readings from Task 1. Join with df_tickets on server_id. Among anomaly readings — what % have an open P1 ticket on the same server?

In [56]:
# Sort data
df_metrics_sorted = df_metrics.sort_values(
    ["server_id", "timestamp"]
)

# Rolling mean
df_metrics_sorted["cpu_roll_mean"] = (
    df_metrics_sorted.groupby(
        "server_id",
        observed=True
    )["cpu_pct"]
    .transform(
        lambda x: x.rolling(window=12).mean()
    )
)

# Rolling std
df_metrics_sorted["cpu_roll_std"] = (
    df_metrics_sorted.groupby(
        "server_id",
        observed=True
    )["cpu_pct"]
    .transform(
        lambda x: x.rolling(window=12).std()
    )
)

# Z-score
df_metrics_sorted["cpu_zscore"] = (
    (
        df_metrics_sorted["cpu_pct"]
        -
        df_metrics_sorted["cpu_roll_mean"]
    )
    /
    df_metrics_sorted["cpu_roll_std"]
)

# Create anomaly flag
df_metrics_sorted["is_anomaly"] = (
    df_metrics_sorted["cpu_zscore"].abs() > 1.5
)

print(
    df_metrics_sorted[
        ["server_id", "cpu_pct", "cpu_zscore", "is_anomaly"]
    ].head()
)

    server_id  cpu_pct  cpu_zscore  is_anomaly
0      srv-01     49.6         NaN       False
500    srv-01     49.6         NaN       False
5      srv-01     82.0         NaN       False
505    srv-01     82.0         NaN       False
10     srv-01     96.6         NaN       False


In [63]:
# Get anomaly rows
cpu_anomalies = df_metrics_sorted[
    df_metrics_sorted["is_anomaly"] == True
]

# Filter open P1 tickets
p1_open_tickets = df_tickets[
    (df_tickets["priority"] == 1 ) &
    (df_tickets["resolved"] == False )
]

# Join datasets
joined = cpu_anomalies.merge(
    p1_open_tickets,
    on="server_id",
    how="left"
)

# Flag matching tickets
joined["has_open_p1"] = (
    joined["ticket_id"].notna()
)

# Percentage calculation
percentage = (
    joined["has_open_p1"].mean()
    * 100
)

print(
    f"Percentage with open P1 ticket: "
    f"{percentage:.2f}%"
)

Percentage with open P1 ticket: 100.00%
